In [ ]:
# Install SHAP for Explainable AI
!pip install shap

# Mount Google Drive to access your dataset directly
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
import numpy as np

# Point to your dataset inside your mounted Google Drive
data_path = "/content/drive/MyDrive/AI_UHI_Project/data/training_dataset.csv"
df = pd.read_csv(data_path)

# Data Cleaning
df.dropna(inplace=True)
df.drop_duplicates(inplace=True)

# Enforce quality bounds
df = df[(df['NDVI'] >= -1.0) & (df['NDVI'] <= 1.0)]
df = df[(df['NDBI'] >= -1.0) & (df['NDBI'] <= 1.0)]
df = df[(df['LST'] >= 20.0) & (df['LST'] <= 60.0)]

print(f"Cleaned dataset shape: {df.shape}")

Cleaned dataset shape: (59801, 9)


In [ ]:
from sklearn.model_selection import train_test_split

# Section 3.2 (A): Feature Engineering
# Extract features (X) and label array (y)
X = df[['NDVI', 'NDBI', 'NDWI', 'DEM', 'LULC']]
y = df['Heat_Class']

# Split data: 80% for training the trees, 20% for testing model accuracy
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training feature set size: {X_train.shape}")
print(f"Testing feature set size: {X_test.shape}")

Training feature set size: (47840, 5)
Testing feature set size: (11961, 5)


In [ ]:
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report
import joblib

# 1. Calculate class weights to hand directly to XGBoost without generating synthetic data
# This protects our true low/medium heat pixel accuracy
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

classes = np.unique(y_train)
weights = compute_class_weight(class_weight='balanced', classes=classes, y=y_train)
class_weights_dict = dict(zip(classes, weights))

# Map weights to the training sample array
sample_weights = np.array([class_weights_dict[cls] for cls in y_train])

# 2. Initialize XGBoost with tuned parameters for multi-class spatial arrays
xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=8,                # Deeper trees to isolate specific urban pockets
    learning_rate=0.05,          # Lower learning rate prevents over-correcting
    subsample=0.8,              # Use 80% of data per tree to prevent overfitting
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='mlogloss',
    n_jobs=-1
)

print("Training high-performance XGBoost model with sample array weights...")
xgb_model.fit(X_train, y_train, sample_weight=sample_weights)

# 3. Evaluate on untouched test data
y_pred_xgb = xgb_model.predict(X_test)
accuracy_xgb = accuracy_score(y_test, y_pred_xgb)

print(f"\n✅ XGBoost Training Complete!")
print(f"XGBoost Test Accuracy: {accuracy_xgb * 100:.2f}%")
print("\n--- XGBoost Optimized Classification Performance ---")
print(classification_report(y_test, y_pred_xgb, target_names=['Low Heat', 'Medium Heat', 'High Heat']))

# 4. Save this model as your primary production asset
model_save_path = "/content/drive/MyDrive/AI_UHI_Project/models/random_forest.pkl" # Overwriting the hook your dashboard expects
joblib.dump(xgb_model, model_save_path)
print(f"Advanced model successfully saved to Drive at: {model_save_path}")

Training high-performance XGBoost model with sample array weights...

✅ XGBoost Training Complete!
XGBoost Test Accuracy: 65.66%

--- XGBoost Optimized Classification Performance ---
              precision    recall  f1-score   support

    Low Heat       0.77      0.69      0.73      6971
 Medium Heat       0.59      0.63      0.60      4807
   High Heat       0.09      0.29      0.13       183

    accuracy                           0.66     11961
   macro avg       0.48      0.53      0.49     11961
weighted avg       0.69      0.66      0.67     11961

Advanced model successfully saved to Drive at: /content/drive/MyDrive/AI_UHI_Project/models/random_forest.pkl


In [ ]:
import shap
import matplotlib.pyplot as plt

# Section 3.3: Explainable AI Insights
# Sample a random slice of test data for fast cloud SHAP computation
X_sample = X_test.sample(1000, random_state=42)

# Instantiate the tree explainer matrix
explainer = shap.TreeExplainer(rf_model)
shap_values = explainer.shap_values(X_sample)

# Generate and save the feature contribution chart for your presentation slides
print("Generating SHAP Feature Interaction Plot...")
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_sample, plot_type="bar", show=False)
plt.title("SHAP Feature Importance: Drivers of Ahmedabad's UHI Effect", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report
import joblib

# 1. Load Data & Engineer Features
data_path = "/content/drive/MyDrive/AI_UHI_Project/data/training_dataset.csv"
df = pd.read_csv(data_path)
df.dropna(inplace=True)
df.drop_duplicates(inplace=True)

# Hotspot Threshold
mean_temp = df['LST'].mean()
std_temp = df['LST'].std()
hotspot_threshold = mean_temp + std_temp
df['UHI_Hotspot'] = np.where(df['LST'] > hotspot_threshold, 1, 0)

# Feature Engineering
df['Urban_Severity_Index'] = df['NDBI'] - df['NDVI']

X = df[['NDVI', 'NDBI', 'NDWI', 'DEM', 'LULC', 'Urban_Severity_Index']]
y = df['UHI_Hotspot']

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Class Weighting
raw_ratio = float(np.sum(y_train == 0)) / np.sum(y_train == 1)
balanced_weight = np.sqrt(raw_ratio)

# ---------------------------------------------------------
# 2. HYPERPARAMETER TUNING (The Maximum Squeeze)
# ---------------------------------------------------------
print("Starting Hyperparameter Tuning... (This may take a few minutes)")

# Define the grid of parameters to test
param_grid = {
    'n_estimators': [300, 500, 700],
    'max_depth': [7, 9, 11],
    'learning_rate': [0.01, 0.03, 0.05],
    'gamma': [0.1, 0.3, 0.5],
    'reg_lambda': [1.0, 2.0, 5.0]
}

# Initialize base model
base_xgb = XGBClassifier(
    scale_pos_weight=balanced_weight,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

# Run the randomized search (tests 20 random combinations from the grid)
random_search = RandomizedSearchCV(
    base_xgb,
    param_distributions=param_grid,
    n_iter=20,
    scoring='f1_macro',
    cv=3,
    verbose=1,
    random_state=42,
    n_jobs=-1
)

random_search.fit(X_train, y_train)

# 3. Extract the absolute best model
best_xgb_model = random_search.best_estimator_

print("\n🎯 Best Parameters Found:")
print(random_search.best_params_)

# 4. Evaluate the optimized master model
y_pred = best_xgb_model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"\n✅ Tuning Complete!")
print(f"Absolute Max Model Accuracy: {accuracy * 100:.2f}%")
print("\n--- Ultimate Classification Performance ---")
print(classification_report(y_test, y_pred, target_names=['Non-Hotspot', 'UHI Hotspot']))

# Save Production Model
model_save_path = "/content/drive/MyDrive/AI_UHI_Project/models/random_forest.pkl"
joblib.dump(best_xgb_model, model_save_path)
print(f"Max-optimized production model saved to Drive at: {model_save_path}")

Starting Hyperparameter Tuning... (This may take a few minutes)
Fitting 3 folds for each of 20 candidates, totalling 60 fits

🎯 Best Parameters Found:
{'reg_lambda': 5.0, 'n_estimators': 500, 'max_depth': 7, 'learning_rate': 0.03, 'gamma': 0.3}

✅ Tuning Complete!
Absolute Max Model Accuracy: 82.21%

--- Ultimate Classification Performance ---
              precision    recall  f1-score   support

 Non-Hotspot       0.89      0.90      0.89      9925
 UHI Hotspot       0.48      0.44      0.46      2036

    accuracy                           0.82     11961
   macro avg       0.68      0.67      0.67     11961
weighted avg       0.82      0.82      0.82     11961

Max-optimized production model saved to Drive at: /content/drive/MyDrive/AI_UHI_Project/models/random_forest.pkl
